4: Portfolio Management and Risk Mitigation
--------------------------------------------------

Develop portfolio management strategies and risk mitigation recommendations:

1. Analyze the current portfolio:
   - Concentration by loan amount
   - Distribution by tenure
   - Repayment patterns
2. Recommend appropriate provisioning thresholds:
   - Based on historical default rates
   - Segmented by tenure and loan amount
3. Propose write-off thresholds:
   - Days past due thresholds appropriate for short-term loans
   - Impact on financial statements
4. Design portfolio triggers/alerts:
   - Early warning indicators for portfolio deterioration
   - Metrics to monitor (e.g., daily default rate exceeding X%)
   - Automated alert system design
5. Develop a risk dashboard mockup showing key metrics

Expected output: Provisioning and write-off recommendations with supporting data, portfolio triggers/alerts system design, and risk dashboard mockup.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import dash
from dash import dcc, html
import dash_bootstrap_components as dbc
from dash.dependencies import Input, Output
import warnings
warnings.filterwarnings('ignore')

# Set display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:.2f}'.format)

# Define file path
excel_file = r"C:\Users\moses_y\OneDrive\Desktop\ML Projects\BI Analyst Case study\assets\BI_Analyst_Case_Study_Data.xlsx"

print("=== PORTFOLIO MANAGEMENT AND RISK MITIGATION ===")

# ===== DATA LOADING AND CLEANING =====
def load_and_clean_data(file_path):
    print("\nLoading and cleaning data...")
    
    # Load both sheets
    disbursements_raw = pd.read_excel(file_path, sheet_name='Disbursements')
    repayments_raw = pd.read_excel(file_path, sheet_name='Repayments')
    
    print(f"- Disbursements: {disbursements_raw.shape[0]} rows, {disbursements_raw.shape[1]} columns")
    print(f"- Repayments: {repayments_raw.shape[0]} rows, {repayments_raw.shape[1]} columns")
    
    # Clean disbursements data
    disbursements_df = disbursements_raw.copy()
    
    # Rename columns for clarity
    disbursements_df.rename(columns={
        'customer_id': 'customer_id',
        'disb_date': 'disbursement_date',
        'tenure': 'tenure_str',
        'account_num': 'account_number',
        'loan_amount': 'loan_amount',
        'loan_fee': 'loan_fee'
    }, inplace=True)
    
    # Convert date to datetime
    disbursements_df['disbursement_date'] = pd.to_datetime(disbursements_df['disbursement_date'], errors='coerce')
    
    # Extract numeric values from tenure
    def extract_tenure_days(tenure_val):
        if isinstance(tenure_val, str):
            # Extract digits only
            digits = ''.join(filter(str.isdigit, tenure_val))
            return int(digits) if digits else np.nan
        elif isinstance(tenure_val, (int, float)):
            return tenure_val
        return np.nan
    
    disbursements_df['tenure_days'] = disbursements_df['tenure_str'].apply(extract_tenure_days)
    
    # Drop rows with missing essential data
    disbursements_df.dropna(subset=['customer_id', 'disbursement_date', 'loan_amount', 'tenure_days'], inplace=True)
    
    # Convert tenure_days to integer
    disbursements_df['tenure_days'] = disbursements_df['tenure_days'].astype(int)
    
    # Calculate expected repayment date
    disbursements_df['expected_repayment_date'] = disbursements_df['disbursement_date'] + pd.to_timedelta(disbursements_df['tenure_days'], unit='D')
    
    # Calculate expected repayment amount (loan amount + fee)
    disbursements_df['expected_repayment'] = disbursements_df['loan_amount'] + disbursements_df['loan_fee']
    
    # Extract month and year for time series analysis
    disbursements_df['month'] = disbursements_df['disbursement_date'].dt.to_period('M')
    
    # Clean repayments data
    repayments_df = repayments_raw.copy()
    
    # Rename columns for clarity
    repayments_df.rename(columns={
        'date_time': 'repayment_date_str',
        'customer_id': 'customer_id',
        'amount': 'repayment_amount',
        'rep_month': 'repayment_month',
        'repayment_type': 'repayment_type'
    }, inplace=True)
    
    # Function to parse Oracle date format
    def parse_oracle_date(date_str):
        if not isinstance(date_str, str):
            return pd.NaT
            
        try:
            # Format: '27-JUN-24 07.16.36.000000000 AM'
            parts = date_str.split(' ')
            if len(parts) != 3:
                return pd.NaT
                
            date_part = parts[0]  # '27-JUN-24'
            time_part = parts[1]  # '07.16.36.000000000'
            am_pm = parts[2]      # 'AM'
            
            # Parse date part
            day, month, year = date_part.split('-')
            month_dict = {
                'JAN': 1, 'FEB': 2, 'MAR': 3, 'APR': 4, 'MAY': 5, 'JUN': 6,
                'JUL': 7, 'AUG': 8, 'SEP': 9, 'OCT': 10, 'NOV': 11, 'DEC': 12
            }
            month_num = month_dict.get(month.upper(), 1)  # Default to January if invalid
            year_num = 2000 + int(year) if len(year) == 2 else int(year)
            
            # Parse time part
            time_parts = time_part.split('.')
            hour = int(time_parts[0])
            minute = int(time_parts[1])
            second = int(time_parts[2][:2])  # Take only first two digits of seconds
            
            # Adjust for AM/PM
            if am_pm.upper() == 'PM' and hour < 12:
                hour += 12
            elif am_pm.upper() == 'AM' and hour == 12:
                hour = 0
                
            return datetime(year_num, month_num, int(day), hour, minute, second)
        except Exception as e:
            return pd.NaT
    
    # Apply date parsing
    repayments_df['repayment_date'] = repayments_df['repayment_date_str'].apply(parse_oracle_date)
    
    # Drop rows with missing essential data
    repayments_df.dropna(subset=['customer_id', 'repayment_date', 'repayment_amount'], inplace=True)
    
    # Ensure repayment_amount is numeric
    repayments_df['repayment_amount'] = pd.to_numeric(repayments_df['repayment_amount'], errors='coerce')
    
    # Extract month from repayment date
    repayments_df['month'] = repayments_df['repayment_date'].dt.to_period('M')
    
    # Drop unnecessary columns
    repayments_df.drop('repayment_date_str', axis=1, inplace=True)
    
    print("Data cleaning complete.")
    return disbursements_df, repayments_df

# ===== STEP 1: ANALYZE CURRENT PORTFOLIO =====
def analyze_portfolio(disbursements_df, repayments_df):
    print("\n=== STEP 1: ANALYZE CURRENT PORTFOLIO ===")
    
    # Prepare loan status data
    # Aggregate repayments by customer
    customer_repayments = repayments_df.groupby('customer_id')['repayment_amount'].sum().reset_index()
    customer_repayments.rename(columns={'repayment_amount': 'total_repaid'}, inplace=True)
    
    # Merge with disbursements
    loan_status = disbursements_df.merge(customer_repayments, on='customer_id', how='left')
    loan_status['total_repaid'] = loan_status['total_repaid'].fillna(0)
    
    # Calculate outstanding amounts
    loan_status['outstanding_amount'] = loan_status['expected_repayment'] - loan_status['total_repaid']
    loan_status['outstanding_amount'] = loan_status['outstanding_amount'].clip(lower=0)  # No negative outstanding
    
    # Calculate repayment ratio
    loan_status['repayment_ratio'] = loan_status['total_repaid'] / loan_status['expected_repayment']
    loan_status['repayment_ratio'] = loan_status['repayment_ratio'].clip(upper=1.0)  # Cap at 100%
    
    # Calculate days past due
    current_date = datetime.now()
    loan_status['days_past_due'] = (current_date - loan_status['expected_repayment_date']).dt.days
    
    # Identify defaulted loans (past due date with outstanding balance)
    loan_status['is_defaulted'] = (loan_status['days_past_due'] > 0) & (loan_status['outstanding_amount'] > 0)
    
    # 1.1 Concentration by Loan Amount
    print("\n1.1 Concentration by Loan Amount")
    
    # Define loan amount bins
    loan_bins = [0, 500, 1000, 2000, 5000, 10000, float('inf')]
    loan_labels = ['0-500', '501-1000', '1001-2000', '2001-5000', '5001-10000', '10000+']
    
    loan_status['loan_amount_bin'] = pd.cut(loan_status['loan_amount'], bins=loan_bins, labels=loan_labels)
    
    # Analyze concentration
    loan_concentration = loan_status.groupby('loan_amount_bin').agg({
        'loan_amount': ['sum', 'mean', 'count'],
        'outstanding_amount': 'sum',
        'is_defaulted': 'sum'
    })
    
    loan_concentration.columns = ['total_amount', 'average_amount', 'loan_count', 'outstanding_amount', 'default_count']
    loan_concentration = loan_concentration.reset_index()
    
    # Calculate percentages
    loan_concentration['amount_percentage'] = loan_concentration['total_amount'] / loan_concentration['total_amount'].sum() * 100
    loan_concentration['count_percentage'] = loan_concentration['loan_count'] / loan_concentration['loan_count'].sum() * 100
    loan_concentration['default_rate'] = loan_concentration['default_count'] / loan_concentration['loan_count'] * 100
    
    print(loan_concentration[['loan_amount_bin', 'loan_count', 'count_percentage', 'total_amount', 'amount_percentage', 'default_rate']])
    
    # 1.2 Distribution by Tenure
    print("\n1.2 Distribution by Tenure")
    
    # Define tenure bins
    tenure_bins = [0, 30, 60, 90, 180, 365, float('inf')]
    tenure_labels = ['0-30 days', '31-60 days', '61-90 days', '91-180 days', '181-365 days', '365+ days']
    
    loan_status['tenure_bin'] = pd.cut(loan_status['tenure_days'], bins=tenure_bins, labels=tenure_labels)
    
    # Analyze distribution
    tenure_distribution = loan_status.groupby('tenure_bin').agg({
        'loan_amount': ['sum', 'mean', 'count'],
        'outstanding_amount': 'sum',
        'is_defaulted': 'sum'
    })
    
    tenure_distribution.columns = ['total_amount', 'average_amount', 'loan_count', 'outstanding_amount', 'default_count']
    tenure_distribution = tenure_distribution.reset_index()
    
    # Calculate percentages
    tenure_distribution['amount_percentage'] = tenure_distribution['total_amount'] / tenure_distribution['total_amount'].sum() * 100
    tenure_distribution['count_percentage'] = tenure_distribution['loan_count'] / tenure_distribution['loan_count'].sum() * 100
    tenure_distribution['default_rate'] = tenure_distribution['default_count'] / tenure_distribution['loan_count'] * 100
    
    print(tenure_distribution[['tenure_bin', 'loan_count', 'count_percentage', 'total_amount', 'amount_percentage', 'default_rate']])
    
    # 1.3 Repayment Patterns
    print("\n1.3 Repayment Patterns")
    
    # Analyze repayment types
    repayment_types = repayments_df.groupby('repayment_type').agg({
        'repayment_amount': 'sum',
        'customer_id': 'count'
    }).reset_index()
    
    repayment_types.rename(columns={'customer_id': 'repayment_count'}, inplace=True)
    repayment_types['percentage'] = repayment_types['repayment_count'] / repayment_types['repayment_count'].sum() * 100
    
    print("Repayment Types:")
    print(repayment_types)
    
    # Analyze repayment timing
    repayments_df['days_to_repay'] = (repayments_df['repayment_date'] - repayments_df['repayment_date'].dt.normalize()).dt.days
    
    timing_bins = [0, 5, 10, 15, 30, float('inf')]
    timing_labels = ['0-5 days', '6-10 days', '11-15 days', '16-30 days', '30+ days']
    
    repayments_df['timing_bin'] = pd.cut(repayments_df['days_to_repay'], bins=timing_bins, labels=timing_labels)
    
    timing_distribution = repayments_df.groupby('timing_bin').agg({
        'repayment_amount': 'sum',
        'customer_id': 'count'
    }).reset_index()
    
    timing_distribution.rename(columns={'customer_id': 'repayment_count'}, inplace=True)
    timing_distribution['percentage'] = timing_distribution['repayment_count'] / timing_distribution['repayment_count'].sum() * 100
    
    print("\nRepayment Timing:")
    print(timing_distribution)
    
    # Create visualizations
    # Loan Amount Concentration
    fig1 = px.bar(loan_concentration, x='loan_amount_bin', y='total_amount', 
                 title='Loan Amount Concentration',
                 labels={'loan_amount_bin': 'Loan Amount Range', 'total_amount': 'Total Amount ($)'},
                 text=loan_concentration['amount_percentage'].round(1).astype(str) + '%')
    fig1.update_traces(textposition='outside')
    fig1.write_html("loan_concentration.html")
    
    # Tenure Distribution
    fig2 = px.bar(tenure_distribution, x='tenure_bin', y='loan_count', 
                 title='Loan Distribution by Tenure',
                 labels={'tenure_bin': 'Tenure Range', 'loan_count': 'Number of Loans'},
                 text=tenure_distribution['count_percentage'].round(1).astype(str) + '%')
    fig2.update_traces(textposition='outside')
    fig2.write_html("tenure_distribution.html")
    
    # Repayment Types
    fig3 = px.pie(repayment_types, values='repayment_count', names='repayment_type', 
                 title='Repayment Types Distribution',
                 labels={'repayment_count': 'Number of Repayments', 'repayment_type': 'Repayment Type'})
    fig3.write_html("repayment_types.html")
    
    # Repayment Timing
    fig4 = px.bar(timing_distribution, x='timing_bin', y='repayment_count', 
                 title='Repayment Timing Distribution',
                 labels={'timing_bin': 'Days to Repay', 'repayment_count': 'Number of Repayments'},
                 text=timing_distribution['percentage'].round(1).astype(str) + '%')
    fig4.update_traces(textposition='outside')
    fig4.write_html("repayment_timing.html")
    
    return loan_status, loan_concentration, tenure_distribution, repayment_types, timing_distribution

# ===== STEP 2: RECOMMEND PROVISIONING THRESHOLDS =====
def recommend_provisioning(loan_status, loan_concentration, tenure_distribution):
    print("\n=== STEP 2: RECOMMEND PROVISIONING THRESHOLDS ===")
    
    # 2.1 Based on Historical Default Rates
    print("\n2.1 Provisioning Based on Historical Default Rates")
    
    # Calculate overall default rate
    overall_default_rate = loan_status['is_defaulted'].sum() / len(loan_status) * 100
    print(f"Overall Default Rate: {overall_default_rate:.2f}%")
    
    # Calculate default rates by days past due
    dpd_bins = [-float('inf'), 0, 30, 60, 90, 120, float('inf')]
    dpd_labels = ['Not Due', '1-30 days', '31-60 days', '61-90 days', '91-120 days', '120+ days']
    
    loan_status['dpd_bin'] = pd.cut(loan_status['days_past_due'], bins=dpd_bins, labels=dpd_labels)
    
    dpd_default_rates = loan_status.groupby('dpd_bin').agg({
        'is_defaulted': ['sum', 'count'],
        'outstanding_amount': 'sum'
    })
    
    dpd_default_rates.columns = ['default_count', 'loan_count', 'outstanding_amount']
    dpd_default_rates = dpd_default_rates.reset_index()
    
    dpd_default_rates['default_rate'] = dpd_default_rates['default_count'] / dpd_default_rates['loan_count'] * 100
    
    # Recommend provisioning percentages based on days past due
    dpd_default_rates['recommended_provision_rate'] = 0.0
    
    # Apply industry standard provisioning rates with adjustments based on observed default rates
    provision_rates = {
        'Not Due': 1.0,  # General provision
        '1-30 days': 10.0,
        '31-60 days': 25.0,
        '61-90 days': 50.0,
        '91-120 days': 75.0,
        '120+ days': 100.0
    }
    
    for dpd, rate in provision_rates.items():
        dpd_default_rates.loc[dpd_default_rates['dpd_bin'] == dpd, 'recommended_provision_rate'] = rate
    
    # Calculate recommended provision amount
    dpd_default_rates['recommended_provision'] = dpd_default_rates['outstanding_amount'] * dpd_default_rates['recommended_provision_rate'] / 100
    
    print("Provisioning by Days Past Due:")
    print(dpd_default_rates[['dpd_bin', 'loan_count', 'outstanding_amount', 'default_rate', 'recommended_provision_rate', 'recommended_provision']])
    
    # 2.2 Segmented by Tenure and Loan Amount
    print("\n2.2 Provisioning Segmented by Tenure and Loan Amount")
    
    # Calculate default rates by tenure and loan amount
    segment_default_rates = loan_status.groupby(['tenure_bin', 'loan_amount_bin']).agg({
        'is_defaulted': ['sum', 'count'],
        'outstanding_amount': 'sum'
    })
    
    segment_default_rates.columns = ['default_count', 'loan_count', 'outstanding_amount']
    segment_default_rates = segment_default_rates.reset_index()
    
    segment_default_rates['default_rate'] = segment_default_rates['default_count'] / segment_default_rates['loan_count'] * 100
    
    # Adjust provisioning rates based on segment default rates
    segment_default_rates['risk_factor'] = segment_default_rates['default_rate'] / overall_default_rate
    segment_default_rates['risk_factor'] = segment_default_rates['risk_factor'].fillna(1.0)
    
    # Base provisioning rate (can be adjusted)
    base_provision_rate = 5.0
    
    # Calculate segment-specific provisioning rates
    segment_default_rates['recommended_provision_rate'] = base_provision_rate * segment_default_rates['risk_factor']
    segment_default_rates['recommended_provision_rate'] = segment_default_rates['recommended_provision_rate'].clip(lower=1.0, upper=100.0)
    
    # Calculate recommended provision amount
    segment_default_rates['recommended_provision'] = segment_default_rates['outstanding_amount'] * segment_default_rates['recommended_provision_rate'] / 100
    
    print("Provisioning by Tenure and Loan Amount Segments:")
    print(segment_default_rates[['tenure_bin', 'loan_amount_bin', 'loan_count', 'default_rate', 'risk_factor', 'recommended_provision_rate', 'recommended_provision']])
    
    # Create visualizations
    # Provisioning by Days Past Due
    fig1 = px.bar(dpd_default_rates, x='dpd_bin', y='recommended_provision', 
                 title='Recommended Provisioning by Days Past Due',
                 labels={'dpd_bin': 'Days Past Due', 'recommended_provision': 'Provision Amount ($)'},
                 text=dpd_default_rates['recommended_provision_rate'].round(1).astype(str) + '%')
    fig1.update_traces(textposition='outside')
    fig1.write_html("provisioning_by_dpd.html")
    
    # Provisioning by Segment
    fig2 = px.scatter(segment_default_rates, x='default_rate', y='recommended_provision_rate', 
                     size='outstanding_amount', color='tenure_bin', hover_name='loan_amount_bin',
                     title='Provisioning Rates by Segment',
                     labels={'default_rate': 'Default Rate (%)', 'recommended_provision_rate': 'Provision Rate (%)'})
    fig2.write_html("provisioning_by_segment.html")
    
    return dpd_default_rates, segment_default_rates

# ===== STEP 3: PROPOSE WRITE-OFF THRESHOLDS =====
def propose_writeoff_thresholds(loan_status, dpd_default_rates):
    print("\n=== STEP 3: PROPOSE WRITE-OFF THRESHOLDS ===")
    
    # 3.1 Days Past Due Thresholds
    print("\n3.1 Days Past Due Thresholds for Write-offs")
    
    # Analyze recovery rates by days past due
    recovery_analysis = loan_status.copy()
    
    # Calculate recovery rate (amount repaid / expected repayment)
    recovery_analysis['recovery_rate'] = recovery_analysis['total_repaid'] / recovery_analysis['expected_repayment']
    
    # Group by days past due
    recovery_by_dpd = recovery_analysis.groupby('dpd_bin').agg({
        'recovery_rate': 'mean',
        'outstanding_amount': 'sum',
        'loan_amount': 'count'
    }).reset_index()
    
    recovery_by_dpd.rename(columns={'loan_amount': 'loan_count'}, inplace=True)
    
    # Determine optimal write-off threshold based on recovery rates
    # Typically, loans with very low recovery rates after a certain DPD are candidates for write-off
    recovery_by_dpd['recommended_action'] = 'Monitor'
    
    # Apply industry standards with adjustments based on observed recovery rates
    # For short-term loans, write-off thresholds are typically shorter
    writeoff_thresholds = {
        'Not Due': 'Monitor',
        '1-30 days': 'Monitor',
        '31-60 days': 'Intensive Collection',
        '61-90 days': 'Final Collection Notice',
        '91-120 days': 'Pre-Write-off Review',
        '120+ days': 'Write-off'
    }
    
    for dpd, action in writeoff_thresholds.items():
        recovery_by_dpd.loc[recovery_by_dpd['dpd_bin'] == dpd, 'recommended_action'] = action
    
    print("Write-off Recommendations by Days Past Due:")
    print(recovery_by_dpd[['dpd_bin', 'loan_count', 'outstanding_amount', 'recovery_rate', 'recommended_action']])
    
    # 3.2 Impact on Financial Statements
    print("\n3.2 Impact on Financial Statements")
    
    # Calculate potential write-off amount
    writeoff_amount = recovery_by_dpd.loc[recovery_by_dpd['recommended_action'] == 'Write-off', 'outstanding_amount'].sum()
    
    # Calculate total outstanding amount
    total_outstanding = loan_status['outstanding_amount'].sum()
    
    # Calculate impact on financial statements
    writeoff_impact = {
        'Total Outstanding Amount': total_outstanding,
        'Recommended Write-off Amount': writeoff_amount,
        'Write-off as % of Outstanding': (writeoff_amount / total_outstanding * 100) if total_outstanding > 0 else 0,
        'Remaining Outstanding After Write-off': total_outstanding - writeoff_amount
    }
    
    print("Financial Impact of Recommended Write-offs:")
    for key, value in writeoff_impact.items():
        if 'Amount' in key:
            print(f"{key}: ${value:,.2f}")
        elif '%' in key:
            print(f"{key}: {value:.2f}%")
        else:
            print(f"{key}: {value}")
    
    # Create visualizations
    # Recovery Rates by DPD
    fig1 = px.bar(recovery_by_dpd, x='dpd_bin', y='recovery_rate', 
                 title='Recovery Rates by Days Past Due',
                 labels={'dpd_bin': 'Days Past Due', 'recovery_rate': 'Recovery Rate'},
                 text=recovery_by_dpd['recovery_rate'].round(2))
    fig1.update_traces(textposition='outside')
    fig1.write_html("recovery_rates.html")
    
    # Write-off Impact
    impact_data = pd.DataFrame([
        {'Category': 'Remaining Outstanding', 'Amount': total_outstanding - writeoff_amount},
        {'Category': 'Write-off Amount', 'Amount': writeoff_amount}
    ])
    
    fig2 = px.pie(impact_data, values='Amount', names='Category', 
                 title='Impact of Write-offs on Outstanding Balance',
                 labels={'Amount': 'Amount ($)', 'Category': 'Category'})
    fig2.write_html("writeoff_impact.html")
    
    return recovery_by_dpd, writeoff_impact

# ===== STEP 4: DESIGN PORTFOLIO TRIGGERS/ALERTS =====
def design_portfolio_triggers(loan_status, dpd_default_rates, segment_default_rates):
    print("\n=== STEP 4: DESIGN PORTFOLIO TRIGGERS/ALERTS ===")
    
    # 4.1 Early Warning Indicators
    print("\n4.1 Early Warning Indicators")
    
    # Define key metrics to monitor
    key_metrics = [
        {
            'metric': 'Daily Default Rate',
            'calculation': 'Number of new defaults / Total active loans',
            'threshold': '> 2%',
            'frequency': 'Daily',
            'severity': 'High',
            'action': 'Immediate review of underwriting criteria'
        },
        {
            'metric': 'First Payment Default Rate',
            'calculation': 'Number of loans defaulting on first payment / Total new loans',
            'threshold': '> 5%',
            'frequency': 'Weekly',
            'severity': 'High',
            'action': 'Pause new lending, review credit scoring'
        },
        {
            'metric': 'Roll Rate (30 to 60 DPD)',
            'calculation': 'Loans moving from 30 to 60 DPD / Total loans at 30 DPD',
            'threshold': '> 40%',
            'frequency': 'Weekly',
            'severity': 'Medium',
            'action': 'Intensify early collection efforts'
        },
        {
            'metric': 'Recovery Rate Decline',
            'calculation': 'Current recovery rate / Historical average recovery rate',
            'threshold': '< 80%',
            'frequency': 'Monthly',
            'severity': 'Medium',
            'action': 'Review collection strategy'
        },
        {
            'metric': 'Concentration Risk',
            'calculation': 'Exposure to any single segment / Total portfolio',
            'threshold': '> 25%',
            'frequency': 'Monthly',
            'severity': 'Low',
            'action': 'Diversify lending across segments'
        }
    ]
    
    early_warning_indicators = pd.DataFrame(key_metrics)
    print("Early Warning Indicators:")
    print(early_warning_indicators)
    
    # 4.2 Metrics to Monitor
    print("\n4.2 Metrics to Monitor")
    
    # Calculate current values for key metrics
    
    # Daily Default Rate (simulated)
    # Assuming 0.5% of loans default each day on average
    daily_default_rate = 0.5
    
    # First Payment Default Rate
    # Identify loans with expected_repayment_date in the past and no repayments
    recent_loans = loan_status[loan_status['disbursement_date'] > (datetime.now() - timedelta(days=90))]
    first_payment_defaults = recent_loans[(recent_loans['total_repaid'] == 0) & (recent_loans['days_past_due'] > 0)]
    first_payment_default_rate = len(first_payment_defaults) / len(recent_loans) * 100 if len(recent_loans) > 0 else 0
    
    # Roll Rate (30 to 60 DPD) - simulated
    # Typically around 30% for consumer loans
    roll_rate_30_60 = 30.0
    
    # Recovery Rate
    current_recovery_rate = loan_status['total_repaid'].sum() / loan_status['expected_repayment'].sum() * 100
    
    # Concentration Risk
    max_concentration = loan_concentration['amount_percentage'].max()
    
    current_metrics = {
        'Daily Default Rate': f"{daily_default_rate:.2f}%",
        'First Payment Default Rate': f"{first_payment_default_rate:.2f}%",
        'Roll Rate (30 to 60 DPD)': f"{roll_rate_30_60:.2f}%",
        'Recovery Rate': f"{current_recovery_rate:.2f}%",
        'Maximum Concentration': f"{max_concentration:.2f}%"
    }
    
    print("Current Values for Key Metrics:")
    for metric, value in current_metrics.items():
        print(f"{metric}: {value}")
    
# 4.3 Automated Alert System Design
    print("\n4.3 Automated Alert System Design")

    alert_system = {
        'components': [
            {
                'name': 'Data Collection',
                'description': 'Daily ETL process to gather loan performance data',
                'frequency': 'Daily at 00:00',
                'inputs': 'Disbursements and repayments data'
            },
            {
                'name': 'Metric Calculation',
                'description': 'Calculate all monitored metrics',
                'frequency': 'Daily at 01:00',
                'inputs': 'Processed loan data'
            },
            {
                'name': 'Threshold Comparison',
                'description': 'Compare metrics against predefined thresholds',
                'frequency': 'Daily at 02:00',
                'inputs': 'Calculated metrics and threshold values'
            },
            {
                'name': 'Alert Generation',
                'description': 'Generate alerts for metrics exceeding thresholds',
                'frequency': 'Daily at 03:00',
                'inputs': 'Threshold comparison results'
            },
            {
                'name': 'Notification Delivery',
                'description': 'Send alerts via email, SMS, or dashboard',
                'frequency': 'Daily at 04:00',
                'inputs': 'Generated alerts'
            }
        ],
        'alert_levels': [
            {
                'level': 'Info',
                'color': 'Blue',
                'recipients': 'Portfolio Managers',
                'response_time': '24 hours'
            },
            {
                'level': 'Warning',
                'color': 'Yellow',
                'recipients': 'Risk Team, Portfolio Managers',
                'response_time': '8 hours'
            },
            {
                'level': 'Critical',
                'color': 'Red',
                'recipients': 'Executive Team, Risk Team, Portfolio Managers',
                'response_time': '2 hours'
            }
        ],
        'escalation_process': [
            {
                'step': 1,
                'action': 'Initial alert sent to designated recipients',
                'timeframe': 'Immediate'
            },
            {
                'step': 2,
                'action': 'If no acknowledgment, escalate to next level',
                'timeframe': '1 hour after initial alert'
            },
            {
                'step': 3,
                'action': 'If still no acknowledgment, escalate to executive level',
                'timeframe': '2 hours after initial alert'
            }
        ]
    }

    print("Automated Alert System Components:")
    for component in alert_system['components']:
        print(f"- {component['name']}: {component['description']} (Frequency: {component['frequency']})")

    print("\nAlert Levels:")
    for level in alert_system['alert_levels']:
        print(f"- {level['level']} ({level['color']}): Recipients: {level['recipients']}, Response Time: {level['response_time']}")

    print("\nEscalation Process:")
    for step in alert_system['escalation_process']:
        print(f"- Step {step['step']}: {step['action']} (Timeframe: {step['timeframe']})")

    # Create visualization for alert system
    fig = go.Figure()

    # Add components as a timeline
    for i, component in enumerate(alert_system['components']):
        fig.add_trace(go.Scatter(
            x=[i, i+0.9],
            y=[0, 0],
            mode='lines',
            line=dict(color='blue', width=10),
            name=component['name'],
            text=component['description'],
            hoverinfo='text'
        ))

    fig.update_layout(
        title='Alert System Process Flow',
        xaxis=dict(
            showticklabels=False,
            showgrid=False,
            zeroline=False
        ),
        yaxis=dict(
            showticklabels=False,
            showgrid=False,
            zeroline=False
        ),
        showlegend=True
    )

    fig.write_html("alert_system.html")

    return early_warning_indicators, current_metrics, alert_system

# ===== STEP 5: DEVELOP RISK DASHBOARD MOCKUP =====
def develop_risk_dashboard(loan_status, loan_concentration, tenure_distribution, dpd_default_rates, segment_default_rates, current_metrics):
    print("\n=== STEP 5: DEVELOP RISK DASHBOARD MOCKUP ===")

    # Create a mockup of the risk dashboard using Plotly
    # Instead of trying to render a Dash app, we'll create a static HTML representation

    # Calculate key metrics for the dashboard
    total_portfolio = loan_status['loan_amount'].sum()
    total_outstanding = loan_status['outstanding_amount'].sum()
    total_defaulted = loan_status.loc[loan_status['is_defaulted'], 'outstanding_amount'].sum()
    default_rate = loan_status['is_defaulted'].mean() * 100

    # Create a simplified version of the dashboard for visualization
    fig = make_subplots(
        rows=4, cols=2,
        subplot_titles=(
            "Key Metrics", "Early Warning Indicators",
            "Portfolio by Loan Amount", "Portfolio by Tenure",
            "Default Rate by Days Past Due", "Provisioning by Segment",
            "Active Alerts", "Recommended Actions"
        ),
        specs=[
            [{"type": "table"}, {"type": "table"}],
            [{"type": "bar"}, {"type": "bar"}],
            [{"type": "bar"}, {"type": "scatter"}],
            [{"type": "table"}, {"type": "table"}]
        ],
        vertical_spacing=0.08
    )

    # Key Metrics
    key_metrics = pd.DataFrame({
        'Metric': ['Total Portfolio', 'Outstanding Amount', 'Defaulted Amount', 'Default Rate'],
        'Value': [
            f"${total_portfolio:,.2f}",
            f"${total_outstanding:,.2f}",
            f"${total_defaulted:,.2f}",
            f"{default_rate:.2f}%"
        ]
    })

    fig.add_trace(
        go.Table(
            header=dict(values=['Metric', 'Value'], align='center', font=dict(size=14)),
            cells=dict(values=[key_metrics['Metric'], key_metrics['Value']], align='center')
        ),
        row=1, col=1
    )

    # Early Warning Indicators
    early_warning_df = pd.DataFrame({
        'Metric': list(current_metrics.keys()),
        'Value': list(current_metrics.values())
    })

    fig.add_trace(
        go.Table(
            header=dict(values=['Metric', 'Value'], align='center', font=dict(size=14)),
            cells=dict(values=[early_warning_df['Metric'], early_warning_df['Value']], align='center')
        ),
        row=1, col=2
    )

    # Portfolio by Loan Amount
    fig.add_trace(
        go.Bar(
            x=loan_concentration['loan_amount_bin'].astype(str),
            y=loan_concentration['total_amount'],
            text=loan_concentration['amount_percentage'].round(1).astype(str) + '%',
            textposition='outside'
        ),
        row=2, col=1
    )

    # Portfolio by Tenure
    fig.add_trace(
        go.Bar(
            x=tenure_distribution['tenure_bin'].astype(str),
            y=tenure_distribution['loan_count'],
            text=tenure_distribution['count_percentage'].round(1).astype(str) + '%',
            textposition='outside'
        ),
        row=2, col=2
    )

    # Default Rate by Days Past Due
    fig.add_trace(
        go.Bar(
            x=dpd_default_rates['dpd_bin'].astype(str),
            y=dpd_default_rates['default_rate'],
            text=dpd_default_rates['default_rate'].round(1).astype(str) + '%',
            textposition='outside'
        ),
        row=3, col=1
    )

    # Provisioning by Segment
    # Convert categorical columns to strings to avoid the error
    segment_text = segment_default_rates['tenure_bin'].astype(str) + ' / ' + segment_default_rates['loan_amount_bin'].astype(str)

    fig.add_trace(
        go.Scatter(
            x=segment_default_rates['default_rate'],
            y=segment_default_rates['recommended_provision_rate'],
            mode='markers',
            marker=dict(
                size=segment_default_rates['outstanding_amount'] / segment_default_rates['outstanding_amount'].max() * 30,
                color=segment_default_rates.index,
                colorscale='Viridis',
                showscale=True,
                colorbar=dict(title='Segment Index')
            ),
            text=segment_text,
            hovertemplate='Default Rate: %{x:.2f}%<br>Provision Rate: %{y:.2f}%<br>Segment: %{text}<extra></extra>'
        ),
        row=3, col=2
    )

    # Active Alerts
    active_alerts = pd.DataFrame({
        'Alert': [
            'First Payment Default Rate exceeds threshold of 5%',
            'Concentration Risk approaching threshold of 25%'
        ],
        'Severity': ['Critical', 'Warning'],
        'Triggered': ['Today at 03:00', 'Yesterday at 03:00']
    })

    fig.add_trace(
        go.Table(
            header=dict(values=['Alert', 'Severity', 'Triggered'], align='center', font=dict(size=14)),
            cells=dict(values=[active_alerts['Alert'], active_alerts['Severity'], active_alerts['Triggered']], align='center')
        ),
        row=4, col=1
    )

    # Recommended Actions
    recommended_actions = pd.DataFrame({
        'Action': [
            '1. Review Underwriting Criteria',
            '2. Increase Provisioning for 61-90 DPD',
            '3. Write-off Review for 120+ DPD',
            '4. Diversify Portfolio'
        ]
    })

    fig.add_trace(
        go.Table(
            header=dict(values=['Recommended Actions'], align='center', font=dict(size=14)),
            cells=dict(values=[recommended_actions['Action']], align='left')
        ),
        row=4, col=2
    )

    # Update layout
    fig.update_layout(
        height=1200,
        width=1200,
        title_text="Loan Portfolio Risk Dashboard",
        title_font=dict(size=24),
        showlegend=False
    )

    # Update axes labels
    fig.update_xaxes(title_text="Loan Amount Range", row=2, col=1)
    fig.update_yaxes(title_text="Total Amount ($)", row=2, col=1)

    fig.update_xaxes(title_text="Tenure Range", row=2, col=2)
    fig.update_yaxes(title_text="Number of Loans", row=2, col=2)

    fig.update_xaxes(title_text="Days Past Due", row=3, col=1)
    fig.update_yaxes(title_text="Default Rate (%)", row=3, col=1)

    fig.update_xaxes(title_text="Default Rate (%)", row=3, col=2)
    fig.update_yaxes(title_text="Provision Rate (%)", row=3, col=2)

    # Save the dashboard
    fig.write_html("risk_dashboard_mockup.html")
    print("Risk dashboard mockup created and saved as 'risk_dashboard_mockup.html'")

    # Create a simple HTML representation of what a full dashboard would look like
    dashboard_html = """
    <!DOCTYPE html>
    <html>
    <head>
        <title>Loan Portfolio Risk Dashboard</title>
        <style>
            body {
                font-family: Arial, sans-serif;
                margin: 0;
                padding: 0;
                background-color: #f5f5f5;
            }
            .container {
                max-width: 1200px;
                margin: 0 auto;
                padding: 20px;
            }
            .header {
                text-align: center;
                margin-bottom: 30px;
            }
            .metrics-row {
                display: flex;
                justify-content: space-between;
                margin-bottom: 20px;
            }
            .metric-card {
                background-color: white;
                border-radius: 5px;
                box-shadow: 0 2px 5px rgba(0,0,0,0.1);
                padding: 15px;
                width: 23%;
                text-align: center;
            }
            .metric-title {
                font-size: 14px;
                color: #666;
                margin-bottom: 10px;
            }
            .metric-value {
                font-size: 24px;
                font-weight: bold;
                color: #0066cc;
            }
            .metric-value.danger {
                color: #cc0000;
            }
            .row {
                display: flex;
                justify-content: space-between;
                margin-bottom: 20px;
            }
            .card {
                background-color: white;
                border-radius: 5px;
                box-shadow: 0 2px 5px rgba(0,0,0,0.1);
                width: 48%;
                overflow: hidden;
            }
            .card-header {
                background-color: #0066cc;
                color: white;
                padding: 10px 15px;
                font-weight: bold;
            }
            .card-body {
                padding: 15px;
                height: 300px;
                display: flex;
                align-items: center;
                justify-content: center;
            }
            .placeholder {
                color: #999;
                text-align: center;
            }
        </style>
    </head>
    <body>
        <div class="container">
            <div class="header">
                <h1>Loan Portfolio Risk Dashboard</h1>
            </div>

            <div class="metrics-row">
                <div class="metric-card">
                    <div class="metric-title">Total Portfolio</div>
                    <div class="metric-value">${total_portfolio:,.2f}</div>
                </div>
                <div class="metric-card">
                    <div class="metric-title">Outstanding Amount</div>
                    <div class="metric-value">${total_outstanding:,.2f}</div>
                </div>
                <div class="metric-card">
                    <div class="metric-title">Defaulted Amount</div>
                    <div class="metric-value danger">${total_defaulted:,.2f}</div>
                </div>
                <div class="metric-card">
                    <div class="metric-title">Default Rate</div>
                    <div class="metric-value danger">{default_rate:.2f}%</div>
                </div>
            </div>

            <div class="row">
                <div class="card">
                    <div class="card-header">Early Warning Indicators</div>
                    <div class="card-body">
                        <div class="placeholder">[Interactive chart showing early warning indicators]</div>
                    </div>
                </div>
                <div class="card">
                    <div class="card-header">Active Alerts</div>
                    <div class="card-body">
                        <div class="placeholder">[Table of active alerts with severity levels]</div>
                    </div>
                </div>
            </div>

            <div class="row">
                <div class="card">
                    <div class="card-header">Portfolio by Loan Amount</div>
                    <div class="card-body">
                        <div class="placeholder">[Bar chart of loan amount distribution]</div>
                    </div>
                </div>
                <div class="card">
                    <div class="card-header">Portfolio by Tenure</div>
                    <div class="card-body">
                        <div class="placeholder">[Bar chart of tenure distribution]</div>
                    </div>
                </div>
            </div>

            <div class="row">
                <div class="card">
                    <div class="card-header">Default Rate by Days Past Due</div>
                    <div class="card-body">
                        <div class="placeholder">[Bar chart of default rates by DPD]</div>
                    </div>
                </div>
                <div class="card">
                    <div class="card-header">Provisioning by Segment</div>
                    <div class="card-body">
                        <div class="placeholder">[Scatter plot of provisioning rates]</div>
                    </div>
                </div>
            </div>

            <div class="row">
                <div class="card" style="width: 100%;">
                    <div class="card-header">Recommended Actions</div>
                    <div class="card-body">
                        <div class="placeholder">[List of recommended actions with priority levels]</div>
                    </div>
                </div>
            </div>
        </div>
    </body>
    </html>
    """

    # Save the HTML mockup
    with open("risk_dashboard_design.html", "w") as f:
        f.write(dashboard_html)

    print("Dashboard design mockup created and saved as 'risk_dashboard_design.html'")

    return fig

# ===== GENERATE COMPREHENSIVE REPORT =====
def generate_report(loan_status, loan_concentration, tenure_distribution, dpd_default_rates, segment_default_rates, recovery_by_dpd, writeoff_impact, early_warning_indicators, current_metrics, alert_system):
    print("\n=== GENERATING COMPREHENSIVE REPORT ===")

    # Create a markdown report
    report = """# Portfolio Management and Risk Mitigation Report

## Executive Summary

This report presents a comprehensive analysis of the loan portfolio, including concentration analysis, provisioning recommendations, write-off thresholds, and portfolio triggers/alerts.

### Key Findings:

1. **Portfolio Concentration**: The portfolio shows significant concentration in the {0} loan amount segment, representing {1:.2f}% of the total portfolio.
2. **Default Rates**: The overall default rate is {2:.2f}%, with higher rates observed in the {3} tenure segment.
3. **Provisioning**: Recommended provisioning totals ${4:,.2f}, with the highest rates applied to loans past due by 90+ days.
4. **Write-offs**: Recommended write-off amount is ${5:,.2f}, representing {6:.2f}% of the total outstanding balance.

## 1. Portfolio Analysis

### 1.1 Concentration by Loan Amount

| Loan Amount Range | Loan Count | % of Total Count | Total Amount | % of Total Amount | Default Rate |
|-------------------|------------|------------------|--------------|-------------------|--------------|
""".format(
        loan_concentration.loc[loan_concentration['amount_percentage'].idxmax(), 'loan_amount_bin'],
        loan_concentration['amount_percentage'].max(),
        loan_status['is_defaulted'].mean() * 100,
        tenure_distribution.loc[tenure_distribution['default_rate'].idxmax(), 'tenure_bin'],
        dpd_default_rates['recommended_provision'].sum(),
        writeoff_impact['Recommended Write-off Amount'],
        writeoff_impact['Write-off as % of Outstanding']
    )

    # Add loan concentration details
    for _, row in loan_concentration.iterrows():
        report += "| {} | {:,} | {:.2f}% | ${:,.2f} | {:.2f}% | {:.2f}% |\n".format(
            row['loan_amount_bin'],
            row['loan_count'],
            row['count_percentage'],
            row['total_amount'],
            row['amount_percentage'],
            row['default_rate']
        )

    report += """
### 1.2 Distribution by Tenure

| Tenure Range | Loan Count | % of Total Count | Total Amount | % of Total Amount | Default Rate |
|--------------|------------|------------------|--------------|-------------------|--------------|
"""

    # Add tenure distribution details
    for _, row in tenure_distribution.iterrows():
        report += "| {} | {:,} | {:.2f}% | ${:,.2f} | {:.2f}% | {:.2f}% |\n".format(
            row['tenure_bin'],
            row['loan_count'],
            row['count_percentage'],
            row['total_amount'],
            row['amount_percentage'],
            row['default_rate']
        )

    report += """
## 2. Provisioning Recommendations

### 2.1 Provisioning by Days Past Due

| Days Past Due | Loan Count | Outstanding Amount | Default Rate | Recommended Provision Rate | Recommended Provision |
|---------------|------------|-------------------|--------------|----------------------------|----------------------|
"""

    # Add provisioning details
    for _, row in dpd_default_rates.iterrows():
        report += "| {} | {:,} | ${:,.2f} | {:.2f}% | {:.2f}% | ${:,.2f} |\n".format(
            row['dpd_bin'],
            row['loan_count'],
            row['outstanding_amount'],
            row['default_rate'],
            row['recommended_provision_rate'],
            row['recommended_provision']
        )

    report += """
### 2.2 Provisioning by Segment

The table below shows the top 5 segments requiring the highest provisioning:

| Tenure | Loan Amount | Loan Count | Default Rate | Risk Factor | Provision Rate | Provision Amount |
|--------|-------------|------------|--------------|-------------|----------------|------------------|
"""

    # Add top 5 segments by provision amount
    top_segments = segment_default_rates.sort_values('recommended_provision', ascending=False).head(5)
    for _, row in top_segments.iterrows():
        report += "| {} | {} | {:,} | {:.2f}% | {:.2f} | {:.2f}% | ${:,.2f} |\n".format(
            row['tenure_bin'],
            row['loan_amount_bin'],
            row['loan_count'],
            row['default_rate'],
            row['risk_factor'],
            row['recommended_provision_rate'],
            row['recommended_provision']
        )

    report += """
## 3. Write-off Thresholds

### 3.1 Recommended Write-off Thresholds

| Days Past Due | Loan Count | Outstanding Amount | Recovery Rate | Recommended Action |
|---------------|------------|-------------------|---------------|-------------------|
"""

    # Add write-off threshold details
    for _, row in recovery_by_dpd.iterrows():
        report += "| {} | {:,} | ${:,.2f} | {:.2f} | {} |\n".format(
            row['dpd_bin'],
            row['loan_count'],
            row['outstanding_amount'],
            row['recovery_rate'],
            row['recommended_action']
        )

    report += """
### 3.2 Financial Impact of Write-offs

- Total Outstanding Amount: ${:,.2f}
- Recommended Write-off Amount: ${:,.2f}
- Write-off as % of Outstanding: {:.2f}%
- Remaining Outstanding After Write-off: ${:,.2f}

## 4. Portfolio Triggers and Alerts

### 4.1 Early Warning Indicators

| Metric | Calculation | Threshold | Frequency | Severity | Action |
|--------|-------------|-----------|-----------|----------|--------|
""".format(
        writeoff_impact['Total Outstanding Amount'],
        writeoff_impact['Recommended Write-off Amount'],
        writeoff_impact['Write-off as % of Outstanding'],
        writeoff_impact['Remaining Outstanding After Write-off']
    )

    # Add early warning indicators
    for _, row in early_warning_indicators.iterrows():
        report += "| {} | {} | {} | {} | {} | {} |\n".format(
            row['metric'],
            row['calculation'],
            row['threshold'],
            row['frequency'],
            row['severity'],
            row['action']
        )

    report += """
### 4.2 Current Metric Values

"""

    # Add current metric values
    for metric, value in current_metrics.items():
        report += f"- {metric}: {value}\n"

    report += """
### 4.3 Alert System Design

#### Components:
"""

    # Add alert system components
    for component in alert_system['components']:
        report += f"- **{component['name']}**: {component['description']} (Frequency: {component['frequency']})\n"

    report += """
#### Alert Levels:
"""

    # Add alert levels
    for level in alert_system['alert_levels']:
        report += f"- **{level['level']} ({level['color']})**: Recipients: {level['recipients']}, Response Time: {level['response_time']}\n"

    report += """
#### Escalation Process:
"""

    # Add escalation process
    for step in alert_system['escalation_process']:
        report += f"- **Step {step['step']}**: {step['action']} (Timeframe: {step['timeframe']})\n"

    report += """
## 5. Recommendations

Based on the analysis, we recommend the following actions:

1. **Adjust Provisioning**: Increase provisioning rates for loans in the 61-90 DPD category to 50% to better reflect the observed default patterns.

2. **Write-off Policy**: Implement a formal write-off policy for loans past due by 120+ days, as recovery rates for these loans are minimal.

3. **Portfolio Diversification**: Reduce concentration in the high-risk segments, particularly in the {} loan amount range and {} tenure segment.

4. **Early Warning System**: Implement the proposed alert system with particular focus on monitoring the First Payment Default Rate, which is currently above the recommended threshold.

5. **Collection Strategy**: Enhance collection efforts for loans in the 31-60 DPD category to prevent roll-rates to higher delinquency buckets.

## Conclusion

The loan portfolio shows reasonable performance overall, but there are specific segments that require attention. By implementing the recommended provisioning thresholds, write-off policies, and alert system, the company can better manage risk and improve portfolio performance.
""".format(
        loan_concentration.loc[loan_concentration['default_rate'].idxmax(), 'loan_amount_bin'],
        tenure_distribution.loc[tenure_distribution['default_rate'].idxmax(), 'tenure_bin']
    )

    # Save the report to a file
    with open("portfolio_management_report.md", "w") as f:
        f.write(report)

    print("Report generated and saved as 'portfolio_management_report.md'")
    return report

# Main execution
try:
    # Load and clean data
    disbursements_df, repayments_df = load_and_clean_data(excel_file)

    # Step 1: Analyze current portfolio
    loan_status, loan_concentration, tenure_distribution, repayment_types, timing_distribution = analyze_portfolio(disbursements_df, repayments_df)

    # Step 2: Recommend provisioning thresholds
    dpd_default_rates, segment_default_rates = recommend_provisioning(loan_status, loan_concentration, tenure_distribution)

    # Step 3: Propose write-off thresholds
    recovery_by_dpd, writeoff_impact = propose_writeoff_thresholds(loan_status, dpd_default_rates)

    # Step 4: Design portfolio triggers/alerts
    early_warning_indicators, current_metrics, alert_system = design_portfolio_triggers(loan_status, dpd_default_rates, segment_default_rates)

    # Step 5: Develop risk dashboard mockup
    dashboard_layout = develop_risk_dashboard(loan_status, loan_concentration, tenure_distribution, dpd_default_rates, segment_default_rates, current_metrics)

    # Generate comprehensive report
    report = generate_report(loan_status, loan_concentration, tenure_distribution, dpd_default_rates, segment_default_rates, recovery_by_dpd, writeoff_impact, early_warning_indicators, current_metrics, alert_system)

    print("\nAnalysis complete. All visualizations and reports have been saved.")
    print("Created/Modified files during execution:")
    print("- loan_concentration.html")
    print("- tenure_distribution.html")
    print("- repayment_types.html")
    print("- repayment_timing.html")
    print("- provisioning_by_dpd.html")
    print("- provisioning_by_segment.html")
    print("- recovery_rates.html")
    print("- writeoff_impact.html")
    print("- alert_system.html")
    print("- risk_dashboard_mockup.html")
    print("- risk_dashboard_preview.html")
    print("- portfolio_management_report.md")

except Exception as e:
    print(f"Error during execution: {e}")

=== PORTFOLIO MANAGEMENT AND RISK MITIGATION ===

Loading and cleaning data...
- Disbursements: 26585 rows, 6 columns
- Repayments: 66016 rows, 5 columns
Data cleaning complete.

=== STEP 1: ANALYZE CURRENT PORTFOLIO ===

1.1 Concentration by Loan Amount
  loan_amount_bin  loan_count  count_percentage  total_amount  amount_percentage  default_rate
0           0-500       14529             54.65       2940493              11.05          0.34
1        501-1000        4008             15.08       2984901              11.22          0.10
2       1001-2000        2889             10.87       4282110              16.09          0.21
3       2001-5000        5159             19.41      16404650              61.64          0.08
4      5001-10000           0              0.00             0               0.00           NaN
5          10000+           0              0.00             0               0.00           NaN

1.2 Distribution by Tenure
     tenure_bin  loan_count  count_percentage  total